### Assignment 1

In [1]:
import torch

#### Assignment 1.1 — Creation Laboratory
- **Objective**: Master tensor creation functions and choose the appropriate constructor based on semantic intent.
- **Given**: No input data.
- **Build / Do**:
  1. Create a $3 \times 4$ matrix filled with zeros.
  2. Create a $2 \times 5$ matrix filled with float `-3.0`.
  3. Create an integer sequence containing values `5, 10, 15, 20, 25` using `torch.arange`.
  4. Create a 1D tensor of eleven evenly spaced floats from `-1.0` to `1.0` inclusive using `torch.linspace`.
  5. Create a $4 \times 4$ identity matrix using `torch.eye`.
  6. Create a random permutation of integers $0$ through $9$ using `torch.randperm`.
- **Pass Criteria**:
  - Check every tensor with `assert` for exact shape, dtype, and boundary values:
    - Zeros: `shape == (3, 4)`, `dtype == torch.float32`, `(t == 0).all()`.
    - `-3.0` matrix: `shape == (2, 5)`, `(t == -3.0).all()`.
    - Arange: `shape == (5,)`, `dtype == torch.int64`, `t[0] == 5`, `t[-1] == 25`.
    - Linspace: `shape == (11,)`, `t[0] == -1.0`, `t[-1] == 1.0`, `t[5] == 0.0`.
    - Identity: `shape == (4, 4)`, `torch.equal(t.diag(), torch.ones(4))`.
    - Permutation: `shape == (10,)`, `set(t.tolist()) == set(range(10))`.
  - Provide written explanation of when to use `arange` (known step) vs. `linspace` (known endpoint count).

In [32]:
zeros_tensor = torch.zeros(3,4)
float_tensor = torch.full((2,5), -3.0)
five_step = torch.arange(5,30,5)
eleven_points = torch.linspace(-1.0, 1.0, 11).round(decimals=2)
identity = torch.eye(4)
rand_perm = torch.randperm(10)

assert zeros_tensor.shape == (3, 4) and zeros_tensor.dtype == torch.float32 and (zeros_tensor == 0).all()
assert float_tensor.shape == (2, 5) and (float_tensor == -3.0).all()
assert five_step.shape == (5,) and five_step.dtype == torch.int64 and five_step[0] == 5 and five_step[-1] == 25
assert eleven_points.shape == (11,) and eleven_points[0] == -1.0 and eleven_points[-1] == 1.0 and eleven_points[5] == 0.0
assert identity.shape == (4,4) and torch.equal(identity.diag(), torch.ones(4))
assert rand_perm.shape == (10,) and set(rand_perm.tolist()) == set(range(10))


#### Assignment 1.2 — Empirical Distribution Experiment
- **Objective**: Verify the empirical statistical properties and bounds of uniform and normal random distributions.
- **Given**: Two synthetic sample tensors of size $1,000,000$:
  - Uniform: `u = torch.rand(1_000_000)`
  - Normal: `n = torch.randn(1_000_000)`
- **Build / Do**:
  1. Compute the empirical mean, standard deviation, minimum, and maximum for both tensors.
  2. Compare observed values against theoretical expectations:
     - Uniform $\mathcal{U}[0, 1)$: $\mu = 0.5$, $\sigma = \frac{1}{\sqrt{12}} \approx 0.2887$, bounded in $[0, 1)$.
     - Normal $\mathcal{N}(0, 1)$: $\mu = 0.0$, $\sigma = 1.0$, unbounded (min/max typically between $\pm 4.5$ and $\pm 5.5$).
- **Pass Criteria**:
  - Assert that `abs(u.mean().item() - 0.5) < 0.01` and `abs(u.std().item() - 0.2887) < 0.01`.
  - Assert that `u.min() >= 0.0` and `u.max() < 1.0`.
  - Assert that `abs(n.mean().item()) < 0.01` and `abs(n.std().item() - 1.0) < 0.01`.
  - Written explanation of why `rand` and `randn` cannot be used interchangeably.

In [41]:
u = torch.rand(1_000_000)
n = torch.randn(1_000_000)

u_mean = u.mean()
u_std = u.std()
u_min= u.min()
u_max= u.max()
assert abs(u_mean.item() -0.5) < 0.01 and abs(u_std.item() - 0.2887) < 0.01
assert u_min >= 0.0 and u_max < 1.0

n_mean = n.mean()
n_std = n.std()
n_min= n.min()
n_max= n.max()
assert abs(n_mean.item() < 0.01) and abs(n_std.item() -1.0) < 0.01

print(f"torch.rand  -> Min: {u_min:.2f} | Max: {u_max:.2f} | Mean: {u_mean:.4f} | Std: {u_std:.4f}")
print(f"torch.randn -> Min: {n_min:.2f} | Max: {n_max:.2f} | Mean: {n_mean:.4f} | Std: {n_std:.4f}")

torch.rand  -> Min: 0.00 | Max: 1.00 | Mean: 0.5000 | Std: 0.2886
torch.randn -> Min: -4.69 | Max: 4.74 | Mean: 0.0006 | Std: 0.9994


#### Assignment 1.3 — Dtype Precision and Memory Footprint
- **Objective**: Quantify the memory footprint and numerical properties of different tensor data types.
- **Given**: A 1D tensor of ones: `x = torch.ones(1_000_000)`.
- **Build / Do**:
  1. Convert `x` to `torch.float64`, `torch.float16`, `torch.bfloat16`, and `torch.bool`.
  2. For each tensor, calculate theoretical bytes: `t.numel() * t.element_size()`.
  3. Verify against actual storage bytes using `t.untyped_storage().nbytes()`.
  4. Tabulate: Dtype, bytes per element, total megabytes, and valid value range / precision.
- **Pass Criteria**:
  - Table accurately lists: `float64` (8 bytes, ~8 MB), `float32` (4 bytes, ~4 MB), `float16` (2 bytes, ~2 MB), `bfloat16` (2 bytes, ~2 MB), `bool` (1 byte, ~1 MB).
  - Written explanation explaining why `float16`/`bfloat16` save memory but require care with dynamic range (underflow/overflow), and why `bool` is unsuitable for continuous weights.

In [47]:
x = torch.ones(1_000_000)

x_float64 = x.to(torch.float64)
x_float16 = x.to(torch.float16)
x_bfloat16 = x.to(torch.bfloat16)
x_bool = x.to(torch.bool)
print(f"Bytes count for Float64 vs actual bytes: {x_float64.numel() * x_float64.element_size()} vs {x_float64.untyped_storage().nbytes()}")
print(f"Bytes for Float16 vs actual bytes: {x_float16.numel() * x_float16.element_size()} vs {x_float16.untyped_storage().nbytes()}")
print(f"Bytes for Bfloat16 vs actual bytes: {x_bfloat16.numel() * x_bfloat16.element_size()} vs {x_bfloat16.untyped_storage().nbytes()}")
print(f"Bytes for Bool vs actual bytes: {x_bool.numel() * x_bool.element_size()} vs {x_bool.untyped_storage().nbytes()}")

Bytes count for Float64 vs actual bytes: 8000000 vs 8000000
Bytes for Float16 vs actual bytes: 2000000 vs 2000000
Bytes for Bfloat16 vs actual bytes: 2000000 vs 2000000
Bytes for Bool vs actual bytes: 1000000 vs 1000000


#### Assignment 1.4 — Independent Random Streams with `torch.Generator`
- **Objective**: Ensure reproducibility and isolation between different stochastic components using explicit generator state.
- **Given**: Two distinct `torch.Generator` instances: `g1 = torch.Generator()` and `g2 = torch.Generator()`.
- **Build / Do**:
  1. Seed both generators with `123`: `g1.manual_seed(123)` and `g2.manual_seed(123)`.
  2. Draw $t_{1a} = \text{randn}(3, \text{generator}=g_1)$ and $t_{2a} = \text{randn}(3, \text{generator}=g_2)$.
  3. Draw $t_{1b} = \text{randn}(3, \text{generator}=g_1)$ and $t_{2b} = \text{randn}(3, \text{generator}=g_2)$.
- **Pass Criteria**:
  - `assert torch.equal(t1a, t2a)` (generators with the same seed produce identical initial sequences).
  - `assert torch.equal(t1b, t2b)` (subsequent draws remain synchronized).
  - `assert not torch.equal(t1a, t1b)` (successive draws from the same stream advance state and differ).
  - `assert not torch.equal(t2a, t2b)`.

In [50]:
g1 = torch.Generator().manual_seed(123)
g2 = torch.Generator().manual_seed(123)

t1a = torch.randn(3, generator=g1)
t2a = torch.randn(3, generator=g2)

t1b = torch.randn(3, generator=g1)
t2b = torch.randn(3, generator=g2)


assert torch.equal(t1a, t2a)
assert torch.equal(t1b, t2b)
assert not torch.equal(t1a, t1b)
assert not torch.equal(t2a, t2b)

#### Assignment 1.5 — Multi-Modal Memory Budget & Device Placement Audit
- **Objective**: Construct a multi-modal dataset pipeline, quantify memory savings through dtype downcasting, and verify device placement.
- **Given**:
  - Tabular features: 5,000 samples with 20 features.
  - Class labels: 5,000 integer labels in $\{0, 1, 2, 3\}$.
  - Image batch: 64 color images of shape `[64, 3, 32, 32]`.
- **Build / Do**:
  1. Construct tabular features with `torch.randn` as `float32`.
  2. Construct labels with `torch.randint` as `int64`.
  3. Construct images with `torch.rand` as `float32`.
  4. Downcast images to `float16` (`.half()`) and tabular features to `float64` (`.double()`).
  5. Compute theoretical bytes (`t.numel() * t.element_size()`) for all tensors and assert exact match with `t.untyped_storage().nbytes()`.
  6. Move all tensors to target device (`device = torch.device("cuda" if torch.cuda.is_available() else "cpu")`).
  7. Verify with assertions: all tensors reside on `device`, dtypes are exact, and `torch.isfinite(t).all()` holds.
- **Pass Criteria**:
  - Memory audit asserts:
    - Images (FP32): `64 * 3 * 32 * 32 * 4 == 786432` bytes (~786 KB).
    - Images (FP16): `64 * 3 * 32 * 32 * 2 == 393216` bytes (~393 KB).
    - Tabular (FP64): `5000 * 20 * 8 == 800000` bytes (~800 KB).
    - Labels (INT64): `5000 * 8 == 40000` bytes (~40 KB).
  - All assertions pass without error.

**Exit criterion:** You can choose the correct tensor creation function based on intent, cast dtypes, audit memory footprints, and manage device placement without guessing.

In [63]:
device = "cuda" if torch.cuda.is_available() else "cpu"
g = torch.Generator(device=device).manual_seed(42)

features = torch.randn(5000, 20, generator=g, dtype=torch.float32, device=device)
class_label = torch.randint(0, 4, (5000,), generator=g, dtype=torch.int64, device=device)
images = torch.rand(64, 3, 32, 32, generator=g, dtype=torch.float32, device=device)
images_fp16 = images.half().to(device)
feature_fp64 = features.double().to(device)

print(f"{features.untyped_storage().nbytes()/1024:.1f}")
print(f"{images.untyped_storage().nbytes()/1024:.1f}")
print(f"{images_fp16.untyped_storage().nbytes()/1024:.1f}")


390.6
768.0
384.0
